# Zarr stores: concurrent writes and snapshots on Databricks

Row-level concurrency and `MERGE` semantics exist only on Databricks, so this notebook checks
the transactional behaviour of `gbx_zarr_write` against Unity Catalog managed manifest tables,
and that a store changed behind its table is noticed.
It has one cell per scenario. Each cell asserts its expectation and prints one `PASS` or
`FAIL` line, and the last cell raises if any scenario failed.

| Row | Scenario | Expectation |
|---|---|---|
| 1 | A crash before commit | A task fails while writing files: the store version and its rows are unchanged |
| 2 | 8 concurrent appenders | Eight writers append disjoint days to one array: all land |
| 3 | Two overlapping writers | Exactly one commits; the other raises `ZarrWriteConflict` |
| 4 | Out-of-order appends with a gap | The shape is the maximum extent; the gap reads as the fill value |
| 5 | A shared coordinate chunk | The writer that finds the labels changed rebases and keeps both writers' labels |
| 6 | A snapshot read during an overwrite | The pinned version equals the old values cell for cell |
| 7 | Drift in a registered store | Touching a file behind the table makes the read raise `ZarrStoreDrift`; `gbx_zarr_sync` records it |
| 8 | Vacuum spares base and tagged files | A vacuum a week later deletes the superseded overlay file and nothing else: not the live one, not the tagged one, not the base store |
| 9 | Restore | A store returns to an earlier version cell for cell, and again to the later one; a version vacuum has removed is refused |
| 10 | A manifest that keeps its files as FILE values | On a workspace with the File type preview: appenders land, overlapping writers conflict, and a vacuum and a copy export agree with the reads |

The scenarios create their stores in a scratch schema and a scratch folder named with a
per-run id, and remove both at the end. They run unchanged on a DBR 19 cluster and on
Serverless environment 6.
Row 10 needs the File type preview of Databricks; where the runtime does not have it, the row
prints `SKIP` and the notebook still passes.

In [ ]:
%pip install --quiet --disable-pip-version-check "geobrix[light_dbr19,zarr_dbr19] @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix-0.6.0-py3-none-any.whl"

In [ ]:
%restart_python

## Setup

The scratch schema and folder are set at the top of the next cell. Change them to a schema
you can create tables in and a folder on a Volume.

The install cell above is for a classic cluster (DBR 19). On Serverless environment 6, add the
`light_env6` and `zarr_env6` extras of the GeoBrix wheel to the notebook environment instead.

In [0]:
import json
import os
import pathlib
import shutil
import threading
import time
import uuid
from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd
import xarray as xr

from databricks.labs.gbx.zarr import (
    BLOCKS_SCHEMA,
    ZarrUnsupported,
    ZarrVersionUnavailable,
    ZarrWriteConflict,
    block_to_numpy,
    gbx_zarr_export,
    gbx_zarr_history,
    gbx_zarr_read,
    gbx_zarr_register,
    gbx_zarr_restore,
    gbx_zarr_sync,
    gbx_zarr_tag,
    gbx_zarr_vacuum,
    gbx_zarr_write,
)
# The commit hooks that line writers up (a barrier between a write's files and its commit),
# and the clock a vacuum reads, so rows 8 and 9 can ask it what it would do a week from now.
from databricks.labs.gbx.zarr import _maint, _write

# Scratch locations: a Unity Catalog schema for the manifest tables and a Volume folder for
# the data files. Both are removed in the last cell.
SCHEMA = "geospatial_docs.gbx_zarr_scratch"
ROOT_BASE = "/Volumes/geospatial_docs/geobrix/sample-data/_probe/zarr/txn"

RUN = uuid.uuid4().hex[:8]
ROOT = f"{ROOT_BASE}/{RUN}"
os.makedirs(ROOT, exist_ok=True)
TABLES = []
RESULTS = {}

DIMS = ("time", "latitude", "longitude")
CHUNKS = {"time": 24, "latitude": 5, "longitude": 6}  # one inner chunk holds one day


def make_days(n_days, ny=10, nx=12, seed=0):
    """A small cube of `n_days` days of hourly data: float32 `t2m` and `u10`."""
    hours = 24 * n_days
    rng = np.random.default_rng(seed)
    shape = (hours, ny, nx)
    return xr.Dataset(
        {
            "t2m": (DIMS, (280 + rng.normal(size=shape)).astype("float32"), {"units": "K"}),
            "u10": (DIMS, rng.normal(size=shape).astype("float32"), {"units": "m s-1"}),
        },
        coords={
            "time": ("time", pd.date_range("2023-01-01", periods=hours, freq="h")),
            "latitude": ("latitude", np.linspace(45.0, 36.0, ny), {"units": "degrees_north"}),
            "longitude": ("longitude", np.arange(nx) * 1.0, {"units": "degrees_east"}),
        },
    )


def day(ds, d):
    """Day `d` (1-based) of a cube."""
    return ds.isel(time=slice(24 * (d - 1), 24 * d))


def blocks(*items):
    """A blocks DataFrame from `(variable, start, values)` items; values are decoded values."""
    rows = []
    for name, start, values in items:
        values = np.ascontiguousarray(values)
        rows.append(
            (
                name,
                list(DIMS),
                list(start),
                [a + n for a, n in zip(start, values.shape)],
                None,
                None,
                values.dtype.str,
                list(values.shape),
                values.tobytes(),
                None,
                None,
            )
        )
    return spark.createDataFrame(rows, BLOCKS_SCHEMA)


def with_retry(fn, tries=6):
    """Reads from a Volume can be briefly inconsistent on workers: retry those."""
    for attempt in range(tries):
        try:
            return fn()
        except Exception as e:  # noqa: BLE001
            if attempt == tries - 1 or "FileNotFound" not in str(e):
                raise
            time.sleep(2)


def read_var(table, name, version=None):
    """Variable `name` of a store as one array: NaN (or NaT) where nothing was written."""
    kw = {} if version is None else {"version_as_of": version}
    rows = with_retry(lambda: gbx_zarr_read(spark, table, variables=[name], **kw).collect())
    stops = [r["stop"] for r in rows]
    shape = tuple(max(s[i] for s in stops) for i in range(len(stops[0])))
    first = block_to_numpy(rows[0])
    if first.dtype.kind == "M":
        out = np.full(shape, np.datetime64("NaT"), dtype=first.dtype)
    elif first.dtype.kind == "f":
        out = np.full(shape, np.nan, dtype=first.dtype)
    else:
        out = np.zeros(shape, dtype=first.dtype)
    for r in rows:
        out[tuple(slice(a, b) for a, b in zip(r["start"], r["stop"]))] = block_to_numpy(r)
    return out


def store_version(table):
    return spark.sql(f"DESCRIBE HISTORY {table} LIMIT 1").first()["version"]


def fingerprint(table):
    """Row count and a hash over the live manifest rows (not the `file` column: Spark cannot hash a FILE value)."""
    row = spark.sql(
        "SELECT count(*) AS n, sum(hash(kind, array, key, write_id, offset, length)) AS h "
        f"FROM {table} WHERE tag IS NULL"
    ).first()
    return (row["n"], row["h"])


def commit_rows(table, write_id):
    return spark.sql(
        f"SELECT count(*) AS n FROM {table} WHERE kind = 'commit' AND key = '{write_id}'"
    ).first()["n"]


def new_store(label, data):
    """Create a managed store from `data`; returns `(table, version)`."""
    table = f"{SCHEMA}.zt_{RUN}_{label}"
    TABLES.append(table)
    version = gbx_zarr_write(
        data, table, root=f"{ROOT}/{label}", chunks=CHUNKS, spark=spark
    )
    return table, version


def wait_for(event, thread, what, timeout=900):
    """Wait for `event`; stop early when `thread` has already ended without setting it."""
    deadline = time.time() + timeout
    while not event.wait(2):
        if not thread.is_alive() or time.time() > deadline:
            raise AssertionError(f"{what} never happened")


def race(table, jobs):
    """Run writes on threads, lined up so every one holds its files, from the same base,
    before any commits. `jobs` is `{label: (write_id, mode, data)}`; returns the version
    each write committed or the exception it raised."""
    ids = {wid for wid, _, _ in jobs.values()}
    barrier = threading.Barrier(len(jobs))
    results = {}

    def lined_up(write_id):
        if write_id in ids:
            barrier.wait(timeout=900)

    def run(label, wid, mode, data):
        try:
            results[label] = gbx_zarr_write(
                data, table, mode=mode, write_id=wid, spark=spark
            )
        except Exception as e:  # noqa: BLE001
            results[label] = e

    _write._BEFORE_COMMIT = lined_up
    try:
        threads = [
            threading.Thread(target=run, args=(label, *job)) for label, job in jobs.items()
        ]
        for t in threads:
            t.start()
        for t in threads:
            t.join(timeout=1800)
    finally:
        _write._BEFORE_COMMIT = None
    return results


def vacuum_a_week_later(table, **kw):
    """`gbx_zarr_vacuum` as it would run eight days from now: the files are all young, so a
    vacuum run now keeps them. The clock is put back afterwards."""
    real = _maint._utcnow
    later = datetime.now(timezone.utc) + timedelta(days=8)
    _maint._utcnow = lambda: later
    try:
        return {
            r["file"]: (r["action"], r["reason"])
            for r in gbx_zarr_vacuum(spark, table, **kw).collect()
        }
    finally:
        _maint._utcnow = real


def tree(path):
    """Every file under a folder with its size and a hash of its bytes."""
    import hashlib

    return {
        p.relative_to(path).as_posix(): (p.stat().st_size, hashlib.sha256(p.read_bytes()).hexdigest())
        for p in sorted(pathlib.Path(path).rglob("*"))
        if p.is_file()
    }

print("run id :", RUN)
print("schema :", SCHEMA)
print("folder :", ROOT)
print("spark  :", spark.version)

run id : cfb064ce
schema : geospatial_docs.gbx_zarr_scratch
folder : /Volumes/geospatial_docs/geobrix/sample-data/_probe/zarr/txn/cfb064ce
spark  : 4.2.0


In [0]:
class Skip(Exception):
    """A row that cannot run on this runtime."""


def check(row, title):
    """Run the cell's scenario; print one PASS, FAIL or SKIP line and keep the outcome."""

    def run(scenario):
        try:
            detail = scenario()
            RESULTS[row] = {"title": title, "result": "PASS", "detail": detail}
            print(f"PASS  row {row}: {title} ({detail})")
        except Skip as e:
            RESULTS[row] = {"title": title, "result": "SKIP", "detail": str(e)}
            print(f"SKIP  row {row}: {title} ({e})")
        except Exception as e:  # noqa: BLE001
            RESULTS[row] = {"title": title, "result": "FAIL", "detail": f"{type(e).__name__}: {e}"}
            print(f"FAIL  row {row}: {title}")
            print(f"      {type(e).__name__}: {str(e)[:600]}")
        return scenario

    return run

## Row 1: a crash before commit stays invisible

`gbx_zarr_write` works in two phases: tasks write immutable data files, then one `MERGE`
commits the rows that point at them. Here the task that encodes one chunk raises during the
first phase. Nothing was committed, so the store version, the manifest rows, and the values a
reader sees are all unchanged.

In [0]:
@check(1, "a crash before commit stays invisible")
def _():
    src = make_days(2)
    table, v0 = new_store("crash", src)
    version_before = store_version(table)
    rows_before = fingerprint(table)
    wid = uuid.uuid4().hex
    update = blocks(("t2m", (0, 0, 0), np.ones((24, 10, 12), "float32")))

    failure = None
    _write._FAULT = ("t2m", "c/0/0/1")  # the task that encodes this chunk raises
    try:
        gbx_zarr_write(update, table, mode="update", write_id=wid, spark=spark)
    except Exception as e:  # noqa: BLE001
        failure = e
    finally:
        _write._FAULT = None

    assert failure is not None, "the write should have failed"
    assert "test fault" in str(failure), str(failure)[:300]
    assert store_version(table) == version_before, "the store version moved"
    assert fingerprint(table) == rows_before, "the manifest rows changed"
    assert commit_rows(table, wid) == 0, "a commit row was written"
    np.testing.assert_array_equal(read_var(table, "t2m"), src.t2m.values)
    return f"store version {version_before} before and after, {rows_before[0]} manifest rows unchanged"

PASS  row 1: a crash before commit stays invisible (store version 1 before and after, 26 manifest rows unchanged)


## Row 2: eight concurrent appenders all land

A store holds one day. Eight driver threads each append a different day to it at the same
time. They all change the same array documents and the same time-coordinate chunk, so the
commits conflict in Delta; each writer rebases onto the latest version and retries (up to
twenty attempts, with a jittered wait), and none of their data files is rewritten. The final
store holds the first day plus the eight appended days.

In [0]:
@check(2, "8 concurrent appenders of disjoint days all land")
def _():
    n = 8
    src = make_days(n + 1)
    table, v0 = new_store("append8", day(src, 1))
    ids = {d: uuid.uuid4().hex for d in range(2, n + 2)}
    results = {}

    def appender(d):
        try:
            results[d] = gbx_zarr_write(
                day(src, d), table, mode="append", write_id=ids[d], spark=spark
            )
        except Exception as e:  # noqa: BLE001
            results[d] = e

    threads = [threading.Thread(target=appender, args=(d,)) for d in ids]
    for t in threads:
        t.start()
    for t in threads:
        t.join(timeout=1800)

    failed = {d: f"{type(r).__name__}: {str(r)[:200]}" for d, r in results.items()
              if isinstance(r, Exception)}
    assert len(results) == n and not failed, failed or f"only {len(results)} writers finished"
    versions = sorted(results.values())
    assert len(set(versions)) == n and versions[0] > v0, versions

    time_axis = read_var(table, "time")
    assert time_axis.shape == (24 * (n + 1),), time_axis.shape
    np.testing.assert_array_equal(time_axis, src.time.values)
    for name in ("t2m", "u10"):
        np.testing.assert_array_equal(read_var(table, name), src[name].values)
    commits = spark.sql(
        f"SELECT count(*) AS n FROM {table} WHERE kind = 'commit' AND tag IS NULL"
    ).first()["n"]
    assert commits == n + 1, commits
    return f"{n} appended days landed in {len(set(versions))} separate versions; shape {n + 1} days x 24 h"

PASS  row 2: 8 concurrent appenders of disjoint days all land (8 appended days landed in 8 separate versions; shape 9 days x 24 h)


## Row 3: two overlapping writers give one winner

Two writers each hold their files, planned from the same base version, and then commit at the
same moment. They write the same chunk, so exactly one commit lands. The other raises
`ZarrWriteConflict` naming the key, and nothing of its write becomes visible. Two partial
writes to different cells of one inner chunk overlap in the same way.

In [0]:
@check(3, "two overlapping writers: one wins, the other raises ZarrWriteConflict")
def _():
    src = make_days(2)
    table, v0 = new_store("overlap", src)
    details = []

    def one_round(label, key, make_a, make_b, expect):
        wa, wb = uuid.uuid4().hex, uuid.uuid4().hex
        before = store_version(table)
        res = race(table, {"a": (wa, "update", make_a()), "b": (wb, "update", make_b())})
        won = [k for k, r in res.items() if isinstance(r, int)]
        lost = [k for k, r in res.items() if isinstance(r, ZarrWriteConflict)]
        assert len(won) == 1 and len(lost) == 1, f"{label}: {res}"
        loser = res[lost[0]]
        ids = {"a": wa, "b": wb}
        assert key in loser.keys, loser.keys
        assert "Fix:" in str(loser)
        assert commit_rows(table, ids[lost[0]]) == 0, "the losing write left a commit row"
        assert commit_rows(table, ids[won[0]]) == 1
        assert res[won[0]] > before, "the winner did not add a version"
        region, values = expect[won[0]]
        np.testing.assert_array_equal(read_var(table, "t2m")[region], values)
        details.append(f"{label}: {won[0]} won at version {res[won[0]]}, {key} in the conflict")

    # the same whole chunks
    ones = np.full((24, 10, 12), 1.0, "float32")
    twos = np.full((24, 10, 12), 2.0, "float32")
    whole_day = (slice(0, 24), slice(0, 10), slice(0, 12))
    one_round(
        "same chunks",
        "t2m/c/0/0/0",
        lambda: blocks(("t2m", (0, 0, 0), ones)),
        lambda: blocks(("t2m", (0, 0, 0), twos)),
        {"a": (whole_day, ones), "b": (whole_day, twos)},
    )
    # different cells of one inner chunk (day 2, hours 24-47)
    first = np.full((12, 5, 6), 3.0, "float32")
    second = np.full((12, 5, 6), 4.0, "float32")
    one_round(
        "different cells of one chunk",
        "t2m/c/1/0/0",
        lambda: blocks(("t2m", (24, 0, 0), first)),
        lambda: blocks(("t2m", (36, 0, 0), second)),
        {
            "a": ((slice(24, 36), slice(0, 5), slice(0, 6)), first),
            "b": ((slice(36, 48), slice(0, 5), slice(0, 6)), second),
        },
    )
    return "; ".join(details)

PASS  row 3: two overlapping writers: one wins, the other raises ZarrWriteConflict (same chunks: b won at version 2, t2m/c/0/0/0 in the conflict; different cells of one chunk: a won at version 3, t2m/c/1/0/0 in the conflict)


## Row 4: out-of-order appends with a gap

On a regular time axis an append lands at its step. Appending day 4 to a store that holds
day 1 skips days 2 and 3: the array grows to the maximum extent and the gap reads as the fill
value (`NaN` for the data, `NaT` for the unwritten time labels). Day 2 is then written into
the gap with an `update`; day 3 stays fill until it is written too.

In [0]:
@check(4, "out-of-order appends with a gap: shape is the maximum, the gap is fill")
def _():
    src = make_days(4)
    table, v0 = new_store("gap", day(src, 1))
    gbx_zarr_write(day(src, 4), table, mode="append", spark=spark)  # days 2 and 3 are skipped

    t2m, time_axis = read_var(table, "t2m"), read_var(table, "time")
    assert t2m.shape == (96, 10, 12) and time_axis.shape == (96,), (t2m.shape, time_axis.shape)
    np.testing.assert_array_equal(t2m[:24], day(src, 1).t2m.values)
    np.testing.assert_array_equal(t2m[72:], day(src, 4).t2m.values)
    assert np.isnan(t2m[24:72]).all(), "the gap is not fill"
    assert np.isnat(time_axis[24:72]).all(), "the gap has time labels"

    gbx_zarr_write(day(src, 2), table, mode="update", spark=spark)  # fills part of the gap
    t2m = read_var(table, "t2m")
    assert t2m.shape == (96, 10, 12), "the shape moved"
    np.testing.assert_array_equal(t2m[24:48], day(src, 2).t2m.values)
    assert np.isnan(t2m[48:72]).all(), "day 3 is no longer fill"

    gbx_zarr_write(day(src, 3), table, mode="update", spark=spark)
    for name in ("t2m", "u10"):
        np.testing.assert_array_equal(read_var(table, name), src[name].values)
    np.testing.assert_array_equal(read_var(table, "time"), src.time.values)
    return "day 4 after day 1: shape 96 h, hours 24-71 fill; days 2 and 3 filled by updates"

PASS  row 4: out-of-order appends with a gap: shape is the maximum, the gap is fill (day 4 after day 1: shape 96 h, hours 24-71 fill; days 2 and 3 filled by updates)


## Row 5: a shared coordinate chunk rebases

Every appender rewrites the one chunk that holds the `time` labels. Here day 2 holds its files
while day 3 appends and commits from the same base. Day 2's commit then finds the array
documents and the coordinate chunk changed: it rebases (re-reads the latest labels, merges its
own, and keeps its data files) and commits. The final labels hold all three days.

In [0]:
@check(5, "a shared coordinate chunk rebases")
def _():
    src = make_days(3)
    table, v0 = new_store("coord", day(src, 1))
    w2, w3 = uuid.uuid4().hex, uuid.uuid4().hex
    landed = {}

    def day_three_commits_first(write_id):
        if write_id == w2 and "v3" not in landed:
            landed["v3"] = gbx_zarr_write(
                day(src, 3), table, mode="append", write_id=w3, spark=spark
            )

    _write._BEFORE_COMMIT = day_three_commits_first
    try:
        v2 = gbx_zarr_write(day(src, 2), table, mode="append", write_id=w2, spark=spark)
    finally:
        _write._BEFORE_COMMIT = None

    assert v0 < landed["v3"] < v2, (v0, landed, v2)
    np.testing.assert_array_equal(read_var(table, "time"), src.time.values)
    for name in ("t2m", "u10"):
        np.testing.assert_array_equal(read_var(table, name), src[name].values)
    owner = [r["write_id"] for r in spark.sql(
        f"SELECT write_id FROM {table} WHERE kind = 'chunk' AND array = 'time' AND tag IS NULL"
    ).collect()]
    assert owner == [w2], "the time chunk was not rewritten by the rebased writer"
    return f"day 3 committed at version {landed['v3']}, day 2 rebased and committed at {v2}; labels for 3 days"

PASS  row 5: a shared coordinate chunk rebases (day 3 committed at version 2, day 2 rebased and committed at 4; labels for 3 days)


## Row 6: a snapshot read during an overwrite is consistent

An overwrite replaces a whole variable. While it holds its new files and has not committed, a
reader pinned to the version before the overwrite, and a reader of the latest version, both
see the old values. After the overwrite commits, the pinned version still equals the old
values cell for cell, and the latest version holds the new ones.

In [0]:
@check(6, "a snapshot read during an overwrite is consistent")
def _():
    src = make_days(2)
    table, v0 = new_store("snapshot", src)
    old = src.t2m.values
    fresh = src[["t2m"]] * 0 + 7.0
    wid = uuid.uuid4().hex
    reached, release = threading.Event(), threading.Event()
    outcome = {}

    def hold_before_commit(write_id):
        if write_id == wid:
            reached.set()
            release.wait(timeout=900)

    def overwrite():
        try:
            outcome["version"] = gbx_zarr_write(
                fresh, table, mode="overwrite", write_id=wid, spark=spark
            )
        except Exception as e:  # noqa: BLE001
            outcome["error"] = e

    _write._BEFORE_COMMIT = hold_before_commit
    writer = threading.Thread(target=overwrite)
    writer.start()
    try:
        wait_for(reached, writer, "the overwrite reaching its commit")
        # the new files are written and nothing is committed
        np.testing.assert_array_equal(read_var(table, "t2m", version=v0), old)
        np.testing.assert_array_equal(read_var(table, "t2m"), old)
    finally:
        release.set()
        writer.join(timeout=900)
        _write._BEFORE_COMMIT = None

    assert "version" in outcome, outcome
    v1 = outcome["version"]
    assert v1 > v0, (v0, v1)
    np.testing.assert_array_equal(read_var(table, "t2m", version=v0), old)  # cell for cell
    assert (read_var(table, "t2m") == 7.0).all(), "the latest version is not the new values"
    np.testing.assert_array_equal(read_var(table, "u10"), src.u10.values)
    return f"version {v0} reads the old values before and after the overwrite lands at version {v1}"

PASS  row 6: a snapshot read during an overwrite is consistent (version 1 reads the old values before and after the overwrite lands at version 2)


## Row 7: drift in a registered store is detected

A store registered in place can still be changed by other programs. GeoBrix records each
object's modification time and size when it registers the store, and checks them before it
reads the object. Here a store written by `xarray` is registered, then one of its chunk files
is rewritten, as another writer would. The read raises `ZarrStoreDrift` and says to run
`gbx_zarr_sync`. The sync lists the change and records it as a new version of the table, and
the read works again.

In [0]:
@check(7, "drift in a registered store is detected")
def _():
    src = make_days(1)
    base = f"{ROOT}/vendor.zarr"
    src.to_zarr(base, mode="w", zarr_format=3, consolidated=False)  # as a vendor leaves it
    table = f"{SCHEMA}.zt_{RUN}_drift"
    TABLES.append(table)
    gbx_zarr_register(spark, base, table)
    np.testing.assert_array_equal(read_var(table, "t2m"), src.t2m.values)

    t2m = pathlib.Path(base, "t2m")
    chunk = next(p for p in sorted(t2m.rglob("*")) if p.is_file() and p.name != "zarr.json")
    time.sleep(2)  # the file system keeps modification times to the second at best
    chunk.write_bytes(chunk.read_bytes())  # touched behind the table's back

    failure = None
    try:
        read_var(table, "t2m")
    except Exception as e:  # noqa: BLE001
        failure = e
    assert failure is not None, "the read should have refused the touched file"
    text = str(failure)
    assert "changed outside GeoBrix" in text and "gbx_zarr_sync" in text, text[:400]

    key = chunk.relative_to(t2m).as_posix()
    changes = {(r["change"], r["array"], r["key"]) for r in gbx_zarr_sync(spark, table).collect()}
    assert changes == {("changed", "t2m", key)}, changes
    np.testing.assert_array_equal(read_var(table, "t2m"), src.t2m.values)
    assert gbx_zarr_sync(spark, table).count() == 0, "the change was not recorded once"
    return f"rewriting t2m/{key} made the read raise ZarrStoreDrift; gbx_zarr_sync recorded it and the read works again"

[register] listing the objects of /Volumes/geospatial_docs/geobrix/sample-data/_probe/zarr/txn/cfb064ce/vendor.zarr
[register] reading the metadata of /Volumes/geospatial_docs/geobrix/sample-data/_probe/zarr/txn/cfb064ce/vendor.zarr
[register] 5 array(s) found
[register] listed 11 object(s) in 2.9s
[register] committing 11 row(s) to geospatial_docs.gbx_zarr_scratch.zt_cfb064ce_drift
[register] registered geospatial_docs.gbx_zarr_scratch.zt_cfb064ce_drift at version 1: 5 object(s), 5 array(s), 1 group(s) in 8.6s
[register] listing the objects of /Volumes/geospatial_docs/geobrix/sample-data/_probe/zarr/txn/cfb064ce/vendor.zarr
[register] synced geospatial_docs.gbx_zarr_scratch.zt_cfb064ce_drift at version 2: {'changed': 1, 'new': 0, 'deleted': 0, 'both': 0}
[register] listing the objects of /Volumes/geospatial_docs/geobrix/sample-data/_probe/zarr/txn/cfb064ce/vendor.zarr
[register] geospatial_docs.gbx_zarr_scratch.zt_cfb064ce_drift is up to date
PASS  row 7: drift in a registered store i

## Row 8: vacuum spares the base and the tagged files

A vacuum deletes the files GeoBrix wrote that no version still readable needs. Here a store
registered in place gets three updates to the same chunk, with a tag on the first. A week later
the second update's file is the only one nothing needs: the third is live, the first is tagged,
and the vendor's own store is never touched. The tagged version still reads after the vacuum.

In [0]:
@check(8, "vacuum spares base and tagged files")
def _():
    src = make_days(2)
    base = f"{ROOT}/vendor8.zarr"
    overlay = f"{ROOT}/overlay8"
    src.to_zarr(
        base, mode="w", zarr_format=3, consolidated=False,
        encoding={v: {"chunks": (24, 5, 6)} for v in ("t2m", "u10")},
    )
    table = f"{SCHEMA}.zt_{RUN}_vacuum"
    TABLES.append(table)
    gbx_zarr_register(spark, base, table, writable=True, overlay_root=overlay)
    vendor = tree(base)

    def update(value):
        block = np.full((24, 5, 6), value, dtype="float32")
        return gbx_zarr_write(blocks(("t2m", (0, 0, 0), block)), table, mode="update", spark=spark)

    def packs():
        found = pathlib.Path(overlay, "data").rglob("*.zpack")
        return sorted(p.relative_to(overlay).as_posix() for p in found)

    first = update(1.0)
    gbx_zarr_tag(spark, table, "first", version_as_of=first)
    update(2.0)
    update(3.0)
    assert len(packs()) == 3, packs()

    dry = vacuum_a_week_later(table, dry_run=True)
    doomed = [f for f, (action, _) in dry.items() if action == "would_delete"]
    assert len(doomed) == 1 and dry[doomed[0]][1] == "superseded", dry
    assert {reason for _, (action, reason) in dry.items() if action == "kept"} == {"live", "tagged"}, dry
    assert len(packs()) == 3, "a dry run removed files"

    vacuum_a_week_later(table)
    assert doomed[0] not in packs() and len(packs()) == 2, packs()
    assert tree(base) == vendor, "the vendor's store was touched"
    assert (read_var(table, "t2m")[0:24, 0:5, 0:6] == 3.0).all()
    assert (read_var(table, "t2m", version="tag:first")[0:24, 0:5, 0:6] == 1.0).all()
    return "deleted the second update's file only; the live and tagged files and the vendor store are intact"

[register] listing the objects of /Volumes/geospatial_docs/geobrix/sample-data/_probe/zarr/txn/cfb064ce/vendor8.zarr
[register] reading the metadata of /Volumes/geospatial_docs/geobrix/sample-data/_probe/zarr/txn/cfb064ce/vendor8.zarr
[register] 5 array(s) found
[register] listed 25 object(s) in 2.2s
[register] committing 25 row(s) to geospatial_docs.gbx_zarr_scratch.zt_cfb064ce_vacuum
[register] registered geospatial_docs.gbx_zarr_scratch.zt_cfb064ce_vacuum at version 1: 19 object(s), 5 array(s), 1 group(s) in 13.3s
PASS  row 8: vacuum spares base and tagged files (deleted the second update's file only; the live and tagged files and the vendor store are intact)


## Row 9: restore returns the store to an earlier version

A restore commits the changes that make the live rows equal an older version's, so nothing is
copied and the restore itself can be undone. Here an overwrite replaces a variable, a restore to
the first version brings the old values back cell for cell, and a restore to the overwrite's
version brings the new ones back. The history lists all four commits. After a vacuum a week
later, the first version is gone for good and a restore to it is refused.

In [0]:
@check(9, "restore returns the store to an earlier version")
def _():
    src = make_days(2)
    table, v0 = new_store("restore", src)
    old = src.t2m.values
    v1 = gbx_zarr_write(src[["t2m"]] * 0 + 7.0, table, mode="overwrite", spark=spark)
    assert (read_var(table, "t2m") == 7.0).all()

    back = gbx_zarr_restore(spark, table, version_as_of=v0)
    assert back > v1, (v1, back)
    np.testing.assert_array_equal(read_var(table, "t2m"), old)
    np.testing.assert_array_equal(read_var(table, "u10"), src.u10.values)
    forward = gbx_zarr_restore(spark, table, version_as_of=v1)
    assert forward > back, (back, forward)
    assert (read_var(table, "t2m") == 7.0).all()
    modes = [r["mode"] for r in gbx_zarr_history(spark, table).collect()]
    assert modes == ["create", "overwrite", "restore", "restore"], modes

    vacuum_a_week_later(table)
    refused = None
    try:
        gbx_zarr_restore(spark, table, version_as_of=v0)
    except ZarrVersionUnavailable as e:
        refused = e
    assert refused is not None, "a vacuumed version was restored"
    assert (read_var(table, "t2m") == 7.0).all(), "a refused restore changed the store"
    return f"version {v0} restored at {back}, version {v1} again at {forward}; after a vacuum, version {v0} is refused"

PASS  row 9: restore returns the store to an earlier version (version 1 restored at 4, version 2 again at 6; after a vacuum, version 1 is refused)


## Row 10: a manifest that keeps its files as FILE values

With `file_mode="external"` the `file` column of the manifest holds Databricks FILE values
(`FILE EXTERNAL`) that point at the pack files GeoBrix wrote, so Unity Catalog knows which files
each version needs. Everything else behaves as in the earlier rows. Three appenders add days at
the same time, two writers race for one chunk, and after a vacuum a week later the reads and a
copy export still agree. The row needs the File type preview; on a runtime without it the row
prints `SKIP`.

In [0]:
@check(10, "a manifest that keeps its files as FILE values")
def _():
    src = make_days(4)
    table = f"{SCHEMA}.zt_{RUN}_file"
    TABLES.append(table)
    try:
        gbx_zarr_write(
            day(src, 1), table, root=f"{ROOT}/file", chunks=CHUNKS, file_mode="external", spark=spark
        )
    except ZarrUnsupported as e:
        TABLES.remove(table)
        raise Skip(str(e))
    kind = spark.sql(
        f"SELECT typeof(file) AS t FROM {table} WHERE file IS NOT NULL LIMIT 1"
    ).first()["t"]
    assert kind.lower().startswith("file"), kind

    ids = {d: uuid.uuid4().hex for d in (2, 3, 4)}
    results = {}

    def appender(d):
        try:
            results[d] = gbx_zarr_write(
                day(src, d), table, mode="append", write_id=ids[d], spark=spark
            )
        except Exception as e:  # noqa: BLE001
            results[d] = e

    threads = [threading.Thread(target=appender, args=(d,)) for d in ids]
    for t in threads:
        t.start()
    for t in threads:
        t.join(timeout=1800)
    failed = {d: f"{type(r).__name__}: {str(r)[:200]}" for d, r in results.items()
              if isinstance(r, Exception)}
    assert len(results) == 3 and not failed, failed or f"only {len(results)} writers finished"
    for name in ("t2m", "u10"):
        np.testing.assert_array_equal(read_var(table, name), src[name].values)

    ones = np.full((24, 10, 12), 1.0, "float32")
    twos = np.full((24, 10, 12), 2.0, "float32")
    res = race(table, {
        "a": (uuid.uuid4().hex, "update", blocks(("t2m", (0, 0, 0), ones))),
        "b": (uuid.uuid4().hex, "update", blocks(("t2m", (0, 0, 0), twos))),
    })
    won = [k for k, r in res.items() if isinstance(r, int)]
    lost = [k for k, r in res.items() if isinstance(r, ZarrWriteConflict)]
    assert len(won) == 1 and len(lost) == 1, res
    expected = src.t2m.values.copy()
    expected[0:24] = ones if won[0] == "a" else twos
    np.testing.assert_array_equal(read_var(table, "t2m"), expected)

    vacuum_a_week_later(table)
    np.testing.assert_array_equal(read_var(table, "t2m"), expected)
    to = f"{ROOT}/file_export"
    gbx_zarr_export(spark, table, to, method="copy")
    with xr.open_zarr(to, consolidated=True, chunks=None) as ds:
        np.testing.assert_array_equal(ds["t2m"].values, expected)
    return (f"{kind} column; 3 appenders landed, writer {won[0]} won the race for one chunk and "
            f"writer {lost[0]} got ZarrWriteConflict; reads after a vacuum a week later and a copy export agree")

SKIP  row 10: a manifest that keeps its files as FILE values (file_mode='external' needs a workspace with the File type preview enabled, and this runtime does not support FILE values Fix: use file_mode="fuse" (the default))


## Clean up and summary

Drop the scratch tables, remove the scratch folder, print the table of outcomes, and raise if
any scenario failed.

In [0]:
problems = []
for table in TABLES:
    try:
        spark.sql(f"DROP TABLE IF EXISTS {table}")
    except Exception as e:  # noqa: BLE001
        problems.append(f"drop {table}: {e}")
try:
    shutil.rmtree(ROOT)
except FileNotFoundError:
    pass
except Exception as e:  # noqa: BLE001
    problems.append(f"remove {ROOT}: {e}")

left_tables = [r[1] for r in spark.sql(f"SHOW TABLES IN {SCHEMA} LIKE 'zt_{RUN}*'").collect()]
assert not left_tables, f"tables left behind: {left_tables}"
assert not os.path.exists(ROOT), f"folder left behind: {ROOT}"
assert not problems, problems
print(f"removed {len(TABLES)} tables and {ROOT}\n")

for row in sorted(RESULTS):
    r = RESULTS[row]
    print(f"{r['result']:5} row {row}: {r['title']}")
passed = sum(r["result"] == "PASS" for r in RESULTS.values())
skipped = sum(r["result"] == "SKIP" for r in RESULTS.values())
ran = len(RESULTS) - skipped
print(f"\n{passed} of {ran} rows passed" + (f" ({skipped} skipped)" if skipped else ""))
assert len(RESULTS) == 10, f"expected 10 rows, ran {len(RESULTS)}"
assert passed == ran, "some rows failed"

removed 9 tables and /Volumes/geospatial_docs/geobrix/sample-data/_probe/zarr/txn/cfb064ce

PASS  row 1: a crash before commit stays invisible
PASS  row 2: 8 concurrent appenders of disjoint days all land
PASS  row 3: two overlapping writers: one wins, the other raises ZarrWriteConflict
PASS  row 4: out-of-order appends with a gap: shape is the maximum, the gap is fill
PASS  row 5: a shared coordinate chunk rebases
PASS  row 6: a snapshot read during an overwrite is consistent
PASS  row 7: drift in a registered store is detected
PASS  row 8: vacuum spares base and tagged files
PASS  row 9: restore returns the store to an earlier version
SKIP  row 10: a manifest that keeps its files as FILE values

9 of 9 rows passed (1 skipped)


In [0]:
dbutils.notebook.exit(json.dumps({"run": RUN, "passed": passed, "rows": {str(k): v["result"] for k, v in RESULTS.items()}}))